In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotting_utils

In [ ]:
adata = sc.read_h5ad(f'{ACC_DATA_ROOT}/outputs/h5ad_files/inprogress_3.h5ad')

In [ ]:
adata

In [ ]:
plotting_utils.plot_umap(adata, 'annot_4', 10, 8, 10, True)

In [ ]:
cycling = adata[adata.obs['annot_4'] == 'Cycling cells - tumor'].copy()

In [ ]:
plotting_utils.plot_umap(cycling, 'annot_4', 10, 8, 10, True)

In [ ]:
sc.tl.leiden(
    cycling,
    key_added='cycling_leid',
    resolution=0.2
)

In [ ]:
plotting_utils.plot_umap(cycling, 'cycling_leid', 10, 8, 10, True)

In [ ]:
# unass = unass[unass.obs['unass_leid'] != '4'].copy()

In [ ]:
sc.tl.rank_genes_groups(cycling, 'cycling_leid', method='wilcoxon')

In [ ]:
sc.pl.rank_genes_groups(cycling, n_genes = 25, sharey = False)

In [ ]:
df = sc.get.rank_genes_groups_df(cycling, None)

In [ ]:
# Filter and separate each group
# Get unique groups
groups = df['group'].unique()

# Store results for each group
results = {}

for group in groups:
    # Filter for this group
    group_df = df[df['group'] == group].copy()
    
    # Apply filters: pvals_adj < 0.05 and (logfoldchanges > 1 OR logfoldchanges is NaN)
    filtered_df = group_df[
        (group_df['pvals_adj'] < 0.05) & 
        ((group_df['logfoldchanges'] > 1) | (group_df['logfoldchanges'].isna()))
    ].copy()
    
    # Sort by score (highest first)
    filtered_df = filtered_df.sort_values('scores', ascending=False)
    
    # Get top 30 genes
    top_30 = filtered_df.head(30)
    
    results[group] = {
        'top_30': top_30,
        'total_filtered': len(filtered_df)
    }

# Print all gene lists first
print("="*60)
print("GENE LISTS FOR ANNOTATION (TOP 30 PER GROUP)")
print("="*60)

for group in groups:
    print(f"\nGROUP {group} ({results[group]['total_filtered']} genes passing filters):")
    print(', '.join(results[group]['top_30']['names'].tolist()))

# Then print detailed tables
print("\n\n" + "="*60)
print("DETAILED TABLES")
print("="*60)

for group in groups:
    print(f"\n{'='*60}")
    print(f"GROUP {group} - Top 30 Marker Genes")
    print(f"{'='*60}\n")
    print(results[group]['top_30'][['names', 'scores', 'logfoldchanges', 'pvals_adj']].to_string(index=False))

In [ ]:
annotation = {
    '0':'ambigious cbd',
    '1':'Stromal cells - tumor',
    '2':'ambigious cbd'
}

In [ ]:
cycling.obs['temp'] = cycling.obs['cycling_leid'].map(annotation)

In [ ]:
adata.obs = adata.obs.join(cycling.obs[['temp']], how = 'left')
adata.obs['annot_4'] = adata.obs['temp'].combine_first(adata.obs['annot_4'])
del adata.obs['temp']

In [ ]:
plotting_utils.plot_umap(adata, 'annot_4', 10, 8, 10, True)

In [ ]:
# adata

In [ ]:
adata.write_h5ad(f'{ACC_DATA_ROOT}/outputs/h5ad_files/inprogress_3.h5ad', compression='lzf')